# SigAlg's `expectation` method

In [31]:
# If running in Google Colab, uncomment the line below and run this cell first
# !pip install sigalg

The `expectation` method in SigAlg is a class method of the `Operators` class for computing *expectations* of random variables and vectors, both unconditional and conditional versions. The API reference for the method is [here](https://johnmyers-phd.com/sigalg/api/core/#sigalg.core.Operators.expectation){target="_blank"}, which includes precise mathematical definitions that we assume the reader knows. This notebook outlines a few usage examples.

## Unconditional expectations

We begin by defining a sample space $\Omega = \{0,1,2,3,4\}$ and a probability measure $P$ on $\Omega$.

In [32]:
from sigalg.core import ProbabilityMeasure, SampleSpace

Omega = SampleSpace().from_sequence(size=5)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.15,
        2: 0.3,
        3: 0.25,
        4: 0.2,
    }
)

Define a random variable $X: \Omega \to \mathbb{R}$ on the sample space $\Omega$ and set its `probability_measure` attribute to $P$ so that all expectations will be computed relative to $P$.

In [33]:
from sigalg.core import RandomVariable

X = RandomVariable(domain=Omega).from_dict(
    {
        0: 1,
        1: 1,
        2: -3,
        3: 2,
        4: -3,
    }
)
X.probability_measure = P

All expectations in SigAlg are instances of `RandomVariable`. The unconditional expectation `E(X)` is thus a constant random variable whose value is the usual expectation $E(X) = \int_\Omega X \, dP$. The `item` method extracts $E(X)$ from `E(X)`.

In [34]:
from sigalg.core import Operators

E = Operators.expectation

expectation_rv = E(X)
expectation = E(X).item()

print(expectation_rv, "\n")
print(expectation)

Random variable 'E(X)':
        E(X)
sample      
0      -0.75
1      -0.75
2      -0.75
3      -0.75
4      -0.75 

-0.75


## Conditional expectations

Define a $\sigma$-algebra $\mathcal{G}$ on $\Omega$ with atoms $A_0=\{0,1\}$, $A_1 = \{2\}$, $A_2 = \{3,4\}$. 

In [35]:
from sigalg.core import SigmaAlgebra

G = SigmaAlgebra(sample_space=Omega, name="G").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 2,
        4: 2,
    }
)

Compute the conditional expectation $E(X\mid \mathcal{G})$. Notice that $X$ is constant on the atom $A_0$ of $\mathcal{G}$, and hence $E(X\mid \mathcal{G})$ is also constant on this atom.

In [36]:
E(X,G)

Random variable 'E(X|G)':
          E(X|G)
sample          
0       1.000000
1       1.000000
2      -3.000000
3      -0.222222
4      -0.222222

The conditional expectation $E(X\mid \mathcal{G})$ is a linear combination of the indicator functions of the atoms of $\mathcal{G}$, namely

$$
E(X\mid \mathcal{G}) = \sum_{k=0}^2 E(X|_{A_k}) I_{A_k},
$$

where $E(X|_{A_k})$ is the expectation of the random variable $X$ restricted to the atom $A_k$ equipped with the conditional probability measure. We compute this linear combination in the following code cell. Notice that it matches `E(X|G)` from above.

In [37]:
I = RandomVariable.indicator_of

sum([E(X(A)).item() * I(A) for A in G.to_atoms()]).with_name("linear_combo")

Random variable 'linear_combo':
        linear_combo
sample              
0           1.000000
1           1.000000
2          -3.000000
3          -0.222222
4          -0.222222

## Testing properties of expectations

Given a sub-$\sigma$-algebra $\mathcal{H}$ of $\mathcal{G}$, the law of iterated expectation says that

$$
E(X \mid \mathcal{H}) = E(E(X \mid \mathcal{G}), \mathcal{H}).
$$

We verify this equality in the following code cell.

In [38]:
H = SigmaAlgebra(sample_space=Omega, name="H").from_dict(
    {
        0: 0,
        1: 0,
        2: 0,
        3: 1,
        4: 1,
    }
)

expectation = E(X, H)
iterated_expectation = E(E(X, G), H)

print(expectation, "\n")
print(iterated_expectation)

Random variable 'E(X|H)':
          E(X|H)
sample          
0      -1.181818
1      -1.181818
2      -1.181818
3      -0.222222
4      -0.222222 

Random variable 'E(E(X|G)|H)':
        E(E(X|G)|H)
sample             
0         -1.181818
1         -1.181818
2         -1.181818
3         -0.222222
4         -0.222222


Given a second random variable $Y: \Omega \to \mathbb{R}$, linearity of expectation says that

$$
E(aX + bY \mid \mathcal{G}) = a E(X\mid \mathcal{G}) + b E(Y\mid \mathcal{G})
$$

for any scalars $a$ and $b$. We verify this equality in the following code cell.

In [39]:
Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: 1,
        1: -2,
        2: 3,
        3: -5,
        4: 1,
    }
)
Y.probability_measure = P

a = 2
b = -3
expectation = E(a * X + b * Y, G)
linear_combo = a * E(X, G) + b * E(Y, G)

print(expectation.with_name("expectation"), "\n")
print(linear_combo.with_name("linear_combo"))

Random variable 'expectation':
        expectation
sample             
0          4.400000
1          4.400000
2        -15.000000
3          6.555556
4          6.555556 

Random variable 'linear_combo':
        linear_combo
sample              
0           4.400000
1           4.400000
2         -15.000000
3           6.555556
4           6.555556


Given a $\mathcal{G}$-measurable random variable $C: \Omega \to \mathbb{R}$, the pull-out property of expectation says that

$$
E(CX \mid \mathcal{G}) = C E(X\mid \mathcal{G}).
$$

We verify this equality in the next code cell.

In [40]:
C = RandomVariable(domain=Omega, name="C").from_dict(
    {
        0: 2,
        1: 2,
        2: -3,
        3: 1,
        4: 1,
    }
)
C.probability_measure = P

expectation = E(C * X, G)
pull_out = C * E(X, G)

print(expectation, "\n")
print(pull_out)


Random variable 'E((C*X)|G)':
        E((C*X)|G)
sample            
0         2.000000
1         2.000000
2         9.000000
3        -0.222222
4        -0.222222 

Random variable '(C*E(X|G))':
        (C*E(X|G))
sample            
0         2.000000
1         2.000000
2         9.000000
3        -0.222222
4        -0.222222
